In [1]:
# Installing yellowbrick
!pip install yellowbrick

In [2]:
pip install -U scikit-learn 

Note: you may need to restart the kernel to use updated packages.


In [3]:
# install the package needed by the extension
%pip install nb-black



  Using cached nb_black-1.0.7.tar.gz (4.8 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [3 lines of output]
      error in nb_black setup command: 'install_requires' must be a string or iterable of strings containing valid project/version requirement specifiers; Expected semicolon (after name with no version specifier) or end
          yapf >= '0.28'; python_version < '3.6'
               ^
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: Failed to build 'nb-black' when getting requirements to build wheel
Note: you may need to restart the kernel to use updated packages.


In [4]:
import warnings

warnings.filterwarnings("ignore")


# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# to perform PCA
from sklearn.decomposition import PCA

# Libraries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 200)

# to scale the data using z-score
from sklearn.preprocessing import StandardScaler

# to compute distances
from scipy.spatial.distance import cdist

# to perform k-means clustering and compute silhouette scores
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# to visualize the elbow curve and silhouette scores
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer

In [5]:
#Loading dataset
student_Registration = pd.read_csv("studentRegistration.csv")

In [6]:
student_Registration.shape

(32593, 5)

In [7]:
student_Registration.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


In [8]:
student_Registration.tail()

,code_module,code_presentation,id_student,date_registration,date_unregistration
32588,GGG,2014J,2640965,-4.0,NaN
32589,GGG,2014J,2645731,-23.0,NaN
32590,GGG,2014J,2648187,-129.0,NaN
32591,GGG,2014J,2679821,-49.0,101.0
32592,GGG,2014J,2684003,-28.0,NaN


In [9]:
student_Registration.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          32593 non-null  object 
 1   code_presentation    32593 non-null  object 
 2   id_student           32593 non-null  int64  
 3   date_registration    32548 non-null  float64
 4   date_unregistration  10072 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 1.2+ MB


In [10]:
student_Registration.isnull().sum()

code_module                0
code_presentation          0
id_student                 0
date_registration         45
date_unregistration    22521
dtype: int64

In [11]:
student_Registration.describe().T

,count,mean,std,min,25%,50%,75%,max
id_student,32593.0,706687.669131,549167.313855,3733.0,508573.0,590310.0,644453.0,2716795.0
date_registration,32548.0,-69.411300,49.260522,-322.0,-100.0,-57.0,-29.0,167.0
date_unregistration,10072.0,49.757645,82.460890,-365.0,-2.0,27.0,109.0,444.0


In [12]:
student_Registration.fillna(0)

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,0.0
1,AAA,2013J,28400,-53.0,0.0
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,0.0
4,AAA,2013J,32885,-176.0,0.0
...,...,...,...,...,...
32588,GGG,2014J,2640965,-4.0,0.0
32589,GGG,2014J,2645731,-23.0,0.0
32590,GGG,2014J,2648187,-129.0,0.0
32591,GGG,2014J,2679821,-49.0,101.0


In [13]:
 #create unique identifier for course run
student_Registration["code_module_presentation"]=student_Registration[['code_module','code_presentation']].agg('-'.join,axis=1)
student_Registration.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration,code_module_presentation
0,AAA,2013J,11391,-159.0,NaN,AAA-2013J
1,AAA,2013J,28400,-53.0,NaN,AAA-2013J
2,AAA,2013J,30268,-92.0,12.0,AAA-2013J
3,AAA,2013J,31604,-52.0,NaN,AAA-2013J
4,AAA,2013J,32885,-176.0,NaN,AAA-2013J


In [14]:
student_Registration.nunique()

code_module                     7
code_presentation               4
id_student                  28785
date_registration             332
date_unregistration           416
code_module_presentation       22
dtype: int64

In [15]:
#Merging student_Registration data set.
 #create unique identifier for course run
student_Registration["code_module_presentation"]=student_Registration[['code_module','code_presentation']].agg('-'.join,axis=1)
student_Registration.head()


,code_module,code_presentation,id_student,date_registration,date_unregistration,code_module_presentation
0,AAA,2013J,11391,-159.0,NaN,AAA-2013J
1,AAA,2013J,28400,-53.0,NaN,AAA-2013J
2,AAA,2013J,30268,-92.0,12.0,AAA-2013J
3,AAA,2013J,31604,-52.0,NaN,AAA-2013J
4,AAA,2013J,32885,-176.0,NaN,AAA-2013J


In [16]:
# create unique identifier for each student in the course run

student_Registration= student_Registration.astype({'id_student':'string'})
student_Registration["code_module_student_presentation"]=student_Registration[['code_module_presentation','id_student']].agg('-'.join,axis=1)
student_Registration.head()


,code_module,code_presentation,id_student,date_registration,date_unregistration,code_module_presentation,code_module_student_presentation
0,AAA,2013J,11391,-159.0,NaN,AAA-2013J,AAA-2013J-11391
1,AAA,2013J,28400,-53.0,NaN,AAA-2013J,AAA-2013J-28400
2,AAA,2013J,30268,-92.0,12.0,AAA-2013J,AAA-2013J-30268
3,AAA,2013J,31604,-52.0,NaN,AAA-2013J,AAA-2013J-31604
4,AAA,2013J,32885,-176.0,NaN,AAA-2013J,AAA-2013J-32885


In [17]:
student_Registration = student_Registration.drop(columns=['code_module','code_presentation','id_student'])
student_Registration.head()

,date_registration,date_unregistration,code_module_presentation,code_module_student_presentation
0,-159.0,NaN,AAA-2013J,AAA-2013J-11391
1,-53.0,NaN,AAA-2013J,AAA-2013J-28400
2,-92.0,12.0,AAA-2013J,AAA-2013J-30268
3,-52.0,NaN,AAA-2013J,AAA-2013J-31604
4,-176.0,NaN,AAA-2013J,AAA-2013J-32885


#Date_Registration and date_unregistraion features are importent for 25% Early drop out analysis

In [18]:
for col in student_Registration.select_dtypes(include=['object']).columns:
    student_Registration[col] = student_Registration[col].str.strip().str.lower()

In [19]:
for col in student_Registration.select_dtypes(include=['object','float','int']).columns:
    print(col)
    print(student_Registration[col].unique())
    print("_" * 40) 

date_registration
[-159.  -53.  -92.  -52. -176. -110.  -67.  -29.  -33. -179. -103.  -47.
  -59.  -68. -180.  -95. -130.  -50. -107.  -27.  -31. -170.  -62. -100.
 -109.    5.  -43.  -26.  -32.  -99.  -82. -197.  -75.  -96. -195.  -61.
  -37.  -36. -132. -138. -174.  -44.  -16.  -54.  -64.  -46. -117.  -45.
 -162.  -57.  -39. -128.  -74.  -12. -136.  -73.  -51. -169. -172. -108.
 -165.  -58.  -79.    2. -151. -198. -186. -145. -191.  -17.   -8. -141.
  -80. -115.  -83.  -18.  -66. -123. -121. -192. -196.   48.  -71.  -81.
  -60.  -49.  -38. -129.   -4.  -19.  -87. -155. -185.  -23.  -15. -120.
 -156. -127.  -30. -158.  -22.  -91.  -42. -144.  -63.  -35.   20. -102.
 -124. -106. -164. -137.  -65. -148. -113. -116.  -28.  -85.  -56. -101.
  -24.  -94.  -11. -105. -153.  -48.  -25.  -72.  -78.  -41. -150. -143.
 -149. -154.  -89.    3.  -21. -135.  -20.   -5. -134.  -10. -122.  -88.
 -147. -114.  -84.  -14.  -70. -140.  -93. -119.   -7.  -86. -146.   10.
  -55. -111. -227. -131. -255.  -

In [20]:
student_Registration.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
32588    False
32589    False
32590    False
32591    False
32592    False
Length: 32593, dtype: bool

In [21]:
student_Registration.duplicated().any()

np.False_

In [22]:
student_Registration[student_Registration.duplicated(keep=False)]

,date_registration,date_unregistration,code_module_presentation,code_module_student_presentation


### There is no duplicated values in entire dataset

# Next step : will go for univariate and bivariate analysis for categorical and int type data